# Gradient-noise geometry pilot

Run on CPU using **Save & Run All**. The synthetic design and optimizer code are embedded. Download `noise_geometry_results/results.json` and `noise_geometry_results/summary.csv` after the run.

We rotate the two Kronecker factors from coordinate-aligned to correlated while keeping the full covariance eigenvalues, total variance, curvature and seed draws fixed. Each optimizer gets the same seven learning rates and two tuning seeds per regime. Evaluation uses eight separate paired seeds. The Evie variants use fixed mean-normalized gamma 30. The random-basis arm keeps the learned gain spectrum and replaces its directions, while the diagonal arm uses only elementwise noise variance.

Compare the paired loss ratios to Muon across regimes. Ratios below 1 favor the corresponding arm. Equal-time results are measured at the last observed checkpoint within a common horizon per seed, with precomputed gradient noise and a single CPU thread. They measure the pilot update loop and exclude tuning, input generation, and model forward/backward. This is a controlled mechanism test using synthetic gradients, with one matrix parameter and no clipping or weight decay.


In [ ]:
from pathlib import Path
Path('pilot_optimizers.py').write_text('import math\nimport numpy as np\nimport torch\nimport torch.nn as nn\nWEIGHT_DECAY = 0.0\nGRAD_CLIP_NORM = 5.0\nHESS_FREQ = 10\nN_HUTCH = 1\nlog = print\n\n# === SHARED OPTIMIZER BLOCK -- keep identical across all 12 files ===\n# ---------------------------------------------------------------------------\n# This block is the single source of truth for every baseline optimizer used\n# in the ICLR benchmark harness. It is pasted BYTE-FOR-BYTE IDENTICAL into all\n# 12 task files by benchmark12/_build/build.py. Do NOT hand-edit it inside a\n# generated file -- edit benchmark12/_build/frag_shared_optim.py and re-run\n# build.py so the change lands in all 12 at once (see benchmark protocol sec 2).\n#\n# Contract with the rest of the file (defined above this block, in the header):\n#   - torch, torch.nn as nn, math, numpy as np  are imported\n#   - WEIGHT_DECAY            : float  (decoupled weight decay, all optimizers)\n#   - GRAD_CLIP_NORM          : float  (global grad-norm clip, applied by engine)\n#   - HESS_FREQ, N_HUTCH      : int    (Sophia Hutchinson probe cadence / count)\n#   - log(msg)                : function (stdout + logfile)\n#\n# Optimizers provided here: AdaBelief, SGDM (SGD+momentum+Nesterov), Sophia,\n# Muon (hybrid: Newton-Schulz on >=2D params, AdamW on the rest), Lion.\n# AdamW itself is torch.optim.AdamW (stdlib) and is built in build_optimizer().\n#\n# Weight decay convention (benchmark protocol sec 4): decoupled, applied as\n#   p <- p * (1 - lr * lambda)\n# BEFORE the gradient-based update, never as L2-in-loss. Every custom optimizer\n# below implements it that way; torch.optim.AdamW already does.\n# ---------------------------------------------------------------------------\n\n# Optimizer-family hyperparameters (frozen 2026-09-08, benchmark protocol sec 4).\nSOPHIA_BETAS      = (0.965, 0.99)   # (grad EMA, Hessian EMA)\nSOPHIA_RHO        = 0.04\nSOPHIA_EPS        = 1e-8\nSOPHIA_BETA_H     = 0.99            # Hessian EMA used inside set_hessian()\nADABELIEF_BETAS   = (0.9, 0.999)\nADABELIEF_EPS     = 1e-16          # AdaBelief paper default\nSGDM_MOMENTUM     = 0.9\nSGDM_NESTEROV     = True\nLION_BETAS        = (0.9, 0.99)\nMUON_MOMENTUM     = 0.95\nMUON_NESTEROV     = True\nMUON_NS_STEPS     = 5\nMUON_NS_COEFFS    = (3.4445, -4.7750, 2.0315)   # Newton-Schulz quintic (Keller Jordan)\nMUON_UPDATE_SCALE = 0.2            # update <- MUON_UPDATE_SCALE * sqrt(max(fan_out, fan_in)) * orthogonalized\nMUON_ADAMW_BETAS  = (0.9, 0.999)  # AdamW sub-group inside the Muon hybrid\nMUON_ADAMW_EPS    = 1e-8\n\n# --- Shampoo (per-layer, AdamW-norm-grafted) -- the "matrix vs diagonal"\n#     baseline the Evie-KF ablation needs (optimizer specification sec 3). Ungrafted\n#     Shampoo is unstable at this scale; we graft the AdamW update norm.\nSHAMPOO_BETA        = 0.95         # preconditioner EMA (tuned jointly with LR)\nSHAMPOO_EPS         = 1e-6\nSHAMPOO_UPDATE_FREQ = 20           # steps between L^-1/4 / R^-1/4 refreshes\nSHAMPOO_GRAFT_BETAS = (0.9, 0.999) # AdamW graft moments\nSHAMPOO_MAXF        = 1024         # 2-D params with a dim > this fall to the graft\nSHAMPOO_KNOB_GRID   = [0.9, 0.95, 0.99, 0.999]   # `beta`, the second tuned axis\n\n# --- Evie-KF: Kronecker-factored risk-sensitive preconditioner\n#     (optimizer specification sec 4.1; full derivation in evie-kf-full-specification.md).\n#     Numerics are ground-truthed against the validated flat-vector reference in\n#     "the reference optimizer" -- see\n#     benchmark12/tests/test_eviekf.py::test_parity_vs_kaggle_reference.\n#\n#     One class, four flags, four rungs of the claim ladder:\n#       gamma=0        -> AdamW, exactly (bit-identical by construction)\n#       diag_only=True -> Evie-diag  (Sigma diagonal; SDProp-2017 / AdaBelief family)\n#       centered=False -> Evie-KFu   (Sigma = E[gg^T], the UNCENTRED object every\n#                                     published Kronecker method uses: K-FAC,\n#                                     Shampoo, SOAP)  -- the C1 novelty ablation\n#       per_layer_norm -> Evie-KFn   (gains rescaled to unit RMS per layer;\n#                                     the WikiText-2 PoC winner, the main-table entrant)\nEVIEKF_FIXED_GAMMA_M = 30.0\nWD_KNOB_GRID = [1e-5, 3.162e-5, 1e-4, 3.162e-4, 1e-3, 3.162e-3, 1e-2]\nEVIEKF_BETAS       = (0.9, 0.999)\nEVIEKF_EPS         = 1e-8\nEVIEKF_BETA_SIG    = 0.95     # frozen -- noise-covariance EMA\nEVIEKF_WARMUP      = 20       # frozen -- operator off for the first WARMUP steps\nEVIEKF_REFRESH     = 20       # frozen -- steps between eigendecomposition refreshes\nEVIEKF_MAXF        = 1024     # frozen -- a Kronecker factor larger than this uses\n                             #           a diagonal approximation (no eigh). On a\n                             #           CIFAR-stem ResNet-18 the reshaped conv\n                             #           factors are 576/1152/2304/4608, so at this\n                             #           limit layers 2-4 -- nearly all the\n                             #           parameters -- run half-diagonal. A\n                             #           DIAGNOSTIC file may raise it via\n                             #           TASK["eviekf_maxf"] (passed in as\n                             #           build_optimizer(eviekf_maxf=...)); no\n                             #           protocol task does, and the paper must\n                             #           state the limit either way.\nEVIEKF_GAIN_CLIP   = 5.0      # frozen -- per_layer_norm gain cap\nEVIEKF_GAMMA_GRID  = [3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0]  # TUNED: the one knob\n# Fixed gamma for the Phase-2 sweeps, which re-search LR per grid point but do\n# NOT joint-search gamma (that would ~7x the sweep cost). build_optimizer()\'s\n# own default is 100.0, but kaggle_evie_v3.py\'s joint search -- the run whose\n# numbers go in the paper -- pinned the optimum at the TOP of the grid (3000),\n# ~30x higher. The batch-size sweep exists to detect the C1 gap widening at\n# small batch; running Evie\'s arms at a gamma the evidence says is far\n# off-optimum risks masking that effect, so the sweeps use 3000, not 100.\n# (Top-edge-pinned even at PoC scale -- a real joint search is still the right\n# fix if the compute budget allows it; see RUNBOOK sec 5 / sec 7.)\nEVIEKF_SWEEP_GAMMA  = 3000.0\n\n# Diagnostic-only gamma grid for the MEAN-NORMALISED Kronecker arm (EvieKFm,\n# FUTURE_WORK FW-9). Under `/tau` the spectrum carries the absolute noise energy,\n# so the tuned gamma has to absorb it and lands anywhere from 3 to 243,000 across\n# the 12 tasks. Under `/mean(mu*lam)` the spectrum has mean eigenvalue 1, so\n# gamma is genuinely dimensionless and should sit near O(1). Same 7-point\n# half-decade discipline, re-centred -- exactly as LR_GRID_CENTRE differs per\n# optimizer family.\nEVIEKF_MEANNORM_GAMMA_GRID = [0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]\n\n# Optimizer registry. EvieKF (centered=True, per_layer_norm=False) fills the\n# reserved slot -- it is the variant validated in kaggle_evie_v3.py (10 seeds,\n# Holm-corrected: beats AdamW/AdaBelief/Lion/Muon/Shampoo/EvieDiag at p=0.0078,\n# ties its own uncentred ablation EvieKFu). EvieKFn (per_layer_norm=True) stays\n# defined in this file but is no longer selected anywhere -- it was the headline\n# arm of an earlier 5-seed / no-Muon-no-Shampoo run and its per_layer_norm gains\n# >1 are a divergence path plain EvieKF does not have.\n# The ablation ladder (EvieDiag/EvieKFu/EvieKF) runs only in the generated\n# benchmark12/ablation/ files (TASK["opt_set"] == "ablation"). EvieKF is\n# therefore BOTH the main table\'s 8th arm AND one of the 3 ablation arms: the 3\n# ablation hosts (wikitext2, cifar10, finance_tech) compute EvieKF twice -- once\n# in the main file (5 seeds), once in the ablation file (10 seeds, a superset).\n# ~15 extra cells total; a known, accepted redundancy, not de-duplicated here.\n#\n# ORDER MATTERS, for two reasons, and neither is cosmetic:\n#  1. run_sweep() and lr_search() walk this list in order, so a run that is cut\n#     short (a Kaggle session that never comes back, a cluster preemption) yields\n#     a PREFIX of it. EvieKF used to be last, which meant a partial run produced\n#     seven baselines and no Evie -- i.e. nothing. EvieKF and AdamW now come\n#     first, so any partial result already contains the comparison the paper is\n#     about. Cells are keyed by NAME ("opt|unit|seed"), never by index, so\n#     reordering is safe for an in-flight run: nothing already computed is lost\n#     or re-run.\n#  2. AdamW must precede Muon. Muon\'s AdamW sub-group is built with\n#     lr_used.get("AdamW"), which is None until AdamW\'s own LR search has\n#     finished -- ordering Muon first would silently give that sub-group Muon\'s\n#     LR instead. Asserted below so a future edit cannot reintroduce it.\nOPT_NAMES          = ["EvieKF", "AdamW", "AdaBelief", "Muon", "Shampoo",\n                      "Sophia", "Lion", "SGD"]\nABLATION_OPT_NAMES = ["EvieKF", "EvieKFu", "EvieDiag"]   # EvieKF\'s 10-seed cells\n                                                         # here are a superset of\n                                                         # the main run\'s 5\nassert OPT_NAMES.index("AdamW") < OPT_NAMES.index("Muon"), \\\n    "AdamW must be searched before Muon (Muon\'s AdamW sub-group reuses its LR)"\n# Scale-generalization secondary tasks (evie-scalegen-spec.md sec 1): trimmed\n# 4-arm set, 3 seeds, run via benchmark12/scalegen/ (TASK["opt_set"]=="scalegen").\n# Same two ordering rules as OPT_NAMES above: EvieKF/AdamW first (a cut-short\n# run\'s prefix still contains the comparison), AdamW before Muon (its AdamW\n# sub-group reuses AdamW\'s frozen LR). All 4 names already exist in\n# build_optimizer() below -- no new optimizer code.\nSCALEGEN_OPT_NAMES = ["EvieKF", "AdamW", "AdaBelief", "Muon"]\nassert SCALEGEN_OPT_NAMES.index("AdamW") < SCALEGEN_OPT_NAMES.index("Muon"), \\\n    "AdamW must be searched before Muon (Muon\'s AdamW sub-group reuses its LR)"\nHESSIAN_CONSUMERS = {"Sophia"}     # optimizers that need the Hutchinson probe\nNOISE_CONSUMERS   = {"EvieKFmwd", "EvieKFmrb", "EvieKFlag", "EvieKFrb", "MuonEvie", "MuonEvieFix", "EvieDiag", "EvieKFu", "EvieKF", "EvieKFn",\n                     "EvieDiagAbs", "EvieKFm", "EvieKFr"}       # need the\n                                  # split-batch noise hook (engine calls set_noise())\n\n# Optimizers with a SECOND tuned axis, searched JOINTLY with the LR grid (same\n# 7-point discipline, same boundary-extend rule -- optimizer specification sec 4.3.2).\n# Anything not listed here is LR-search only, exactly as before.\nKNOB_GRID = {\n    **{n: list(WD_KNOB_GRID) for n in ("AdamWwd", "AdaBeliefwd", "SGDwd", "Lionwd", "Muonwd")},\n    "EvieKFlag": list(EVIEKF_GAMMA_GRID), "EvieKFrb": list(EVIEKF_GAMMA_GRID),\n    "EvieKFn": list(EVIEKF_GAMMA_GRID), "EvieKF": list(EVIEKF_GAMMA_GRID),\n    "EvieKFu": list(EVIEKF_GAMMA_GRID), "EvieDiag": list(EVIEKF_GAMMA_GRID),\n    "Shampoo": list(SHAMPOO_KNOB_GRID),\n    # diagnostic arms (DERIVATION.md sec 5.1 / sec 11, FW-9 and FW-11)\n    "EvieDiagAbs": list(EVIEKF_GAMMA_GRID),      # same grid as EvieDiag, absolute gate\n    "EvieKFr": list(EVIEKF_GAMMA_GRID),          # same as EvieKF, rescale disabled\n    "EvieKFm": list(EVIEKF_MEANNORM_GAMMA_GRID), # re-centred: gamma is dimensionless here\n    "EvieKFmrb": list(EVIEKF_MEANNORM_GAMMA_GRID), # EvieKFm with the random-basis control (pinned runs only)\n}\n\n# Per-optimizer-family LR grid centre. Every optimizer gets the SAME NUMBER of\n# grid points (7, half-decade spacing); only the centre differs by family\n# because effective LR scale is not comparable across families (benchmark protocol sec 5).\nLR_GRID_CENTRE = {\n    "AdamWwd": 1e-3, "AdaBeliefwd": 1e-3, "SGDwd": 3e-2,\n    "Lionwd": 1e-4, "Muonwd": 2e-2, "EvieKFmwd": 1e-3,\n    "EvieKFlag": 1e-3, "EvieKFrb": 1e-3, "EvieKFmrb": 1e-3,\n    "MuonSub": 1e-3, "MuonEvie": 1e-3, "MuonEvieFix": 1e-3,\n    "AdamW":     1e-3,\n    "AdaBelief": 1e-3,\n    "SGD":       3e-2,\n    "Sophia":    1e-3,\n    "Muon":      2e-2,\n    "Lion":      1e-4,\n    "Shampoo":   1e-3,          # AdamW-grafted -> Adam-family centring\n    "EvieKFn":   1e-3,          # Evie-KF family: P^-1/2 is AdamW\'s own denominator,\n    "EvieKF":    1e-3,          # so the effective LR scale matches the Adam family\n    "EvieKFu":   1e-3,\n    "EvieDiag":  1e-3,\n    "EvieDiagAbs": 1e-3,        # diagnostic arms, same family centring\n    "EvieKFm":   1e-3,\n    "EvieKFr":   1e-3,\n}\n\n\ndef half_decade_grid(centre, n_below=3, n_above=3):\n    """7-point half-decade (sqrt(10)~=3.162x) grid centred on `centre`.\n    Returns a sorted list of length n_below + 1 + n_above."""\n    r = 10.0 ** 0.5\n    pts = [centre * (r ** k) for k in range(-n_below, n_above + 1)]\n    return [float(f"{p:.4g}") for p in pts]\n\n\ndef knob_grid_extend(grid, at_low):\n    """Boundary-extend rule (benchmark protocol sec 5) applied to a second-axis knob grid.\n    Returns up to 2 new points past the edge the search pinned to. gamma-style\n    grids (all values >= 1) extend multiplicatively by 3x; a beta-style grid\n    bounded in (0, 1) densifies toward the bound instead."""\n    g = sorted(float(x) for x in grid)\n    if g and all(0.0 < x < 1.0 for x in g):\n        if at_low:\n            lo, d = g[0], (g[1] - g[0] if len(g) > 1 else g[0] * 0.5)\n            return [round(lo - k * d, 6) for k in (1, 2) if lo - k * d > 0.0]\n        hi = g[-1]\n        return [round(1.0 - (1.0 - hi) / f, 6) for f in (3.0, 10.0)\n                if 0.0 < 1.0 - (1.0 - hi) / f < 1.0]\n    if at_low:\n        return [float(f"{g[0] / f:.4g}") for f in (3.0, 9.0)]\n    return [float(f"{g[-1] * f:.4g}") for f in (3.0, 9.0)]\n\n\n# ---------------------------------------------------------------------------\n#  AdaBelief -- Adam with the second moment tracking (g - m)^2 ("belief" in\n#  the gradient direction) instead of g^2. Decoupled weight decay, bias\n#  correction, no RAdam-style rectification (plain decoupled AdaBelief).\n# ---------------------------------------------------------------------------\nclass AdaBelief(torch.optim.Optimizer):\n    def __init__(self, params, lr=1e-3, betas=ADABELIEF_BETAS, eps=ADABELIEF_EPS,\n                 weight_decay=0.0):\n        if lr <= 0.0:\n            raise ValueError(f"AdaBelief: invalid lr {lr}")\n        defaults = dict(lr=lr, beta1=betas[0], beta2=betas[1], eps=eps,\n                        weight_decay=weight_decay)\n        super().__init__(params, defaults)\n\n    @torch.no_grad()\n    def step(self, closure=None):\n        loss = None\n        if closure is not None:\n            with torch.enable_grad():\n                loss = closure()\n        for group in self.param_groups:\n            lr, b1, b2 = group["lr"], group["beta1"], group["beta2"]\n            eps, wd = group["eps"], group["weight_decay"]\n            for p in group["params"]:\n                if p.grad is None:\n                    continue\n                g = p.grad\n                if g.is_sparse:\n                    raise RuntimeError("AdaBelief does not support sparse gradients")\n                if wd != 0.0:\n                    p.mul_(1.0 - lr * wd)                       # decoupled WD\n                st = self.state[p]\n                if len(st) == 0:\n                    st["step"] = 0\n                    st["m"] = torch.zeros_like(p)\n                    st["s"] = torch.zeros_like(p)               # EMA of (g - m)^2\n                st["step"] += 1\n                t = st["step"]\n                m, s = st["m"], st["s"]\n                m.mul_(b1).add_(g, alpha=1.0 - b1)\n                diff = g - m\n                s.mul_(b2).addcmul_(diff, diff, value=1.0 - b2)\n                s.add_(eps)\n                m_hat = m / (1.0 - b1 ** t)\n                s_hat = s / (1.0 - b2 ** t)\n                p.addcdiv_(m_hat, s_hat.sqrt().add_(eps), value=-lr)\n        return loss\n\n\n# ---------------------------------------------------------------------------\n#  SGDM -- SGD + heavy-ball momentum + Nesterov, with DECOUPLED weight decay\n#  (torch.optim.SGD only offers coupled L2, hence this thin re-implementation\n#  so the WD convention matches every other optimizer in the harness).\n#  Cosine LR schedule is applied by the engine, same as for all optimizers.\n# ---------------------------------------------------------------------------\nclass SGDM(torch.optim.Optimizer):\n    def __init__(self, params, lr=3e-2, momentum=SGDM_MOMENTUM,\n                 nesterov=SGDM_NESTEROV, weight_decay=0.0):\n        if lr <= 0.0:\n            raise ValueError(f"SGDM: invalid lr {lr}")\n        defaults = dict(lr=lr, momentum=momentum, nesterov=nesterov,\n                        weight_decay=weight_decay)\n        super().__init__(params, defaults)\n\n    @torch.no_grad()\n    def step(self, closure=None):\n        loss = None\n        if closure is not None:\n            with torch.enable_grad():\n                loss = closure()\n        for group in self.param_groups:\n            lr, mom = group["lr"], group["momentum"]\n            nesterov, wd = group["nesterov"], group["weight_decay"]\n            for p in group["params"]:\n                if p.grad is None:\n                    continue\n                g = p.grad\n                if wd != 0.0:\n                    p.mul_(1.0 - lr * wd)                       # decoupled WD\n                st = self.state[p]\n                if len(st) == 0:\n                    st["buf"] = torch.zeros_like(p)\n                buf = st["buf"]\n                buf.mul_(mom).add_(g)\n                d = g.add(buf, alpha=mom) if nesterov else buf\n                p.add_(d, alpha=-lr)\n        return loss\n\n\n# ---------------------------------------------------------------------------\n#  Sophia (SophiaH) -- clipped Newton-style update with a Hutchinson diagonal\n#  Hessian EMA. Hyperparameters frozen per benchmark protocol sec 4:\n#  beta1=0.965, beta2(Hessian EMA)=0.99, rho=0.04, eps=1e-8, HESS_FREQ=5,\n#  N_HUTCH=4. The engine calls set_hessian() every HESS_FREQ steps with the\n#  Hutchinson estimate from hutchinson_diag_hessian().\n# ---------------------------------------------------------------------------\nclass Sophia(torch.optim.Optimizer):\n    def __init__(self, params, lr=1e-3, betas=SOPHIA_BETAS, rho=SOPHIA_RHO,\n                 eps=SOPHIA_EPS, weight_decay=0.0):\n        if lr <= 0.0:\n            raise ValueError(f"Sophia: invalid lr {lr}")\n        defaults = dict(lr=lr, beta1=betas[0], beta2=betas[1], rho=rho, eps=eps,\n                        weight_decay=weight_decay)\n        super().__init__(params, defaults)\n        self.last_h_norm = float("nan")\n\n    @torch.no_grad()\n    def set_hessian(self, param_h_pairs, beta_h=SOPHIA_BETA_H):\n        sq = 0.0\n        for p, h in param_h_pairs:\n            st = self.state[p]\n            h_new = h.detach().abs()\n            if "h_hat" not in st:\n                st["h_hat"] = h_new.clone()\n            else:\n                st["h_hat"].mul_(beta_h).add_(h_new, alpha=1.0 - beta_h)\n            sq += float(h_new.pow(2).sum().item())\n        self.last_h_norm = math.sqrt(sq)\n\n    @torch.no_grad()\n    def step(self, closure=None):\n        loss = None\n        if closure is not None:\n            with torch.enable_grad():\n                loss = closure()\n        for group in self.param_groups:\n            lr, b1 = group["lr"], group["beta1"]\n            rho, eps, wd = group["rho"], group["eps"], group["weight_decay"]\n            for p in group["params"]:\n                if p.grad is None:\n                    continue\n                g = p.grad\n                if wd != 0.0:\n                    p.mul_(1.0 - lr * wd)                       # decoupled WD\n                st = self.state[p]\n                if "step" not in st:\n                    st["step"] = 0\n                    st["m"] = torch.zeros_like(p)\n                if "h_hat" not in st:\n                    st["h_hat"] = torch.ones_like(p)\n                st["step"] += 1\n                t = st["step"]\n                m = st["m"]\n                m.mul_(b1).add_(g, alpha=1.0 - b1)\n                m_hat = m / (1.0 - b1 ** t)\n                h = st["h_hat"].clamp(min=eps)\n                update = (m_hat / (rho * h)).clamp_(-1.0, 1.0)\n                p.add_(update, alpha=-lr)\n        return loss\n\n\ndef _spatial_avg(h):\n    """Per-row average for 2D, scalar average for 1D, spatial average for 4D\n    (kept for parity with the historical Hutchinson pipeline)."""\n    if h.dim() == 4:\n        return h.mean(dim=[2, 3], keepdim=True).expand_as(h).clone()\n    if h.dim() == 2:\n        return h.mean(dim=1, keepdim=True).expand_as(h).clone()\n    if h.dim() == 1:\n        return h.mean().expand_as(h).clone()\n    return h.clone()\n\n\ndef hutchinson_diag_hessian(model, params, compute_loss, n_hutch=N_HUTCH,\n                            optimizer=None, feed_optimizer=False):\n    """Hutchinson diagonal-Hessian estimator: for each of n_hutch Rademacher\n    vectors v, form v (x) Hv via a double backward and accumulate; divide by\n    n_hutch, take abs. Feeds Sophia via set_hessian() when feed_optimizer=True.\n    `compute_loss` is a zero-arg closure that returns the (scalar) training loss\n    for the current batch -- the engine passes the domain\'s forward_loss so the\n    probe matches the real loss (e.g. the (B,T,V) reshape for language).\n    Returns the list of per-parameter diagonal estimates."""\n    with torch.enable_grad():\n        loss_h = compute_loss()\n    grads = torch.autograd.grad(loss_h, params, create_graph=True,\n                                retain_graph=True, allow_unused=True)\n    h_acc = [torch.zeros_like(p) for p in params]\n    for k in range(n_hutch):\n        vs = [(torch.randint(0, 2, p.shape, device=p.device, dtype=torch.float32)\n               * 2.0 - 1.0).to(p.dtype) for p in params]\n        gv = sum((g * v).sum() for g, v in zip(grads, vs) if g is not None)\n        retain = (k < n_hutch - 1)\n        hvps = torch.autograd.grad(gv, params, retain_graph=retain, allow_unused=True)\n        for i, (v, hv) in enumerate(zip(vs, hvps)):\n            if hv is not None:\n                h_acc[i].add_(v * hv.detach())\n        del vs, gv, hvps\n    del grads\n    for h in h_acc:\n        h.div_(n_hutch).abs_()\n    if feed_optimizer and optimizer is not None:\n        optimizer.set_hessian(list(zip(params, h_acc)), beta_h=SOPHIA_BETA_H)\n    return h_acc\n\n\n# ---------------------------------------------------------------------------\n#  Lion (EvoLved Sign Momentum) -- update = sign(b1*m + (1-b1)*g); the EMA is\n#  then refreshed with b2. Decoupled weight decay. LR runs ~10x below Adam\'s,\n#  hence the separate grid centre (LR_GRID_CENTRE["Lion"]).\n# ---------------------------------------------------------------------------\nclass Lion(torch.optim.Optimizer):\n    def __init__(self, params, lr=1e-4, betas=LION_BETAS, weight_decay=0.0):\n        if lr <= 0.0:\n            raise ValueError(f"Lion: invalid lr {lr}")\n        defaults = dict(lr=lr, beta1=betas[0], beta2=betas[1],\n                        weight_decay=weight_decay)\n        super().__init__(params, defaults)\n\n    @torch.no_grad()\n    def step(self, closure=None):\n        loss = None\n        if closure is not None:\n            with torch.enable_grad():\n                loss = closure()\n        for group in self.param_groups:\n            lr, b1, b2, wd = (group["lr"], group["beta1"], group["beta2"],\n                              group["weight_decay"])\n            for p in group["params"]:\n                if p.grad is None:\n                    continue\n                g = p.grad\n                if wd != 0.0:\n                    p.mul_(1.0 - lr * wd)                       # decoupled WD\n                st = self.state[p]\n                if len(st) == 0:\n                    st["m"] = torch.zeros_like(p)\n                m = st["m"]\n                update = m.mul(b1).add_(g, alpha=1.0 - b1).sign_()\n                p.add_(update, alpha=-lr)\n                m.mul_(b2).add_(g, alpha=1.0 - b2)\n        return loss\n\n\n# ---------------------------------------------------------------------------\n#  Muon (MomentUm Orthogonalized by Newton-schulz) -- hybrid optimizer.\n#  Params with ndim >= 2 that are NOT flagged as embedding/head (see\n#  split_params_for_muon) get the Muon update: heavy-ball (Nesterov) momentum\n#  followed by Newton-Schulz orthogonalization of the (reshaped-to-2D) update,\n#  scaled by MUON_UPDATE_SCALE * sqrt(max(fan_out, fan_in)). Every other param\n#  (biases, norm params, all 1D params, token embeddings, LM head) gets a\n#  standard decoupled-WD AdamW update. Both sub-groups are carried in ONE\n#  Optimizer so checkpoint/resume and the cosine LR scheduler work unchanged.\n#\n#  Per the frozen decision (benchmark protocol sec 8.2 + user): the 7-point LR search\n#  tunes ONLY the Muon (2D) sub-group\'s LR; the AdamW sub-group\'s LR is fixed\n#  to this file\'s separately-frozen AdamW LR (passed as adamw_lr to\n#  build_optimizer). Conv weights (4D) are reshaped (O, I*kH*kW) and DO go to\n#  the Muon side -- the strict "2D only" reading would leave Muon == AdamW on\n#  the entire vision domain, which is not a meaningful baseline. This split is\n#  written once here and pasted identically into all 12 files.\n# ---------------------------------------------------------------------------\ndef _zeropower_via_newtonschulz5(G, steps=MUON_NS_STEPS, coeffs=MUON_NS_COEFFS,\n                                 eps=1e-7):\n    """Quintic Newton-Schulz iteration -> approximate orthogonalization of G\n    (2D). Operates in float32. Returns a matrix with the same shape as G whose\n    singular values are pushed toward 1."""\n    assert G.ndim == 2, "Newton-Schulz expects a 2D matrix"\n    a, b, c = coeffs\n    X = G.float()\n    X = X / (X.norm() + eps)\n    transpose = X.size(0) > X.size(1)\n    if transpose:\n        X = X.T\n    for _ in range(steps):\n        A = X @ X.T\n        B = b * A + c * (A @ A)\n        X = a * X + B @ X\n    if transpose:\n        X = X.T\n    return X.to(G.dtype)\n\n\n@torch.no_grad()\ndef _muon_matrix_step(opt, group):\n    """Exact shared matrix update used by Muon and MuonEvie."""\n    lr, wd = group["lr"], group["weight_decay"]\n    mom, nesterov, ns_steps = (group["momentum"], group["nesterov"],\n                               group["ns_steps"])\n    for p in group["params"]:\n        if p.grad is None:\n            continue\n        g = p.grad\n        if g.ndim > 2:\n            g = g.reshape(g.size(0), -1)\n        st = opt.state[p]\n        if len(st) == 0:\n            st["buf"] = torch.zeros_like(g)\n        buf = st["buf"]\n        buf.mul_(mom).add_(g)\n        d = g.add(buf, alpha=mom) if nesterov else buf\n        o = _zeropower_via_newtonschulz5(d, steps=ns_steps)\n        fan_out, fan_in = o.size(0), o.size(1)\n        scale = MUON_UPDATE_SCALE * math.sqrt(max(fan_out, fan_in))\n        if wd != 0.0:\n            p.mul_(1.0 - lr * wd)                   # decoupled WD\n        p.add_(o.reshape(p.shape), alpha=-lr * scale)\n\n\nclass Muon(torch.optim.Optimizer):\n    def __init__(self, param_groups):\n        # Each incoming group must set: params, lr, use_muon (bool).\n        defaults = dict(\n            lr=2e-2, use_muon=True,\n            momentum=MUON_MOMENTUM, nesterov=MUON_NESTEROV, ns_steps=MUON_NS_STEPS,\n            betas=MUON_ADAMW_BETAS, eps=MUON_ADAMW_EPS, weight_decay=WEIGHT_DECAY,\n        )\n        super().__init__(param_groups, defaults)\n\n    @torch.no_grad()\n    def step(self, closure=None):\n        loss = None\n        if closure is not None:\n            with torch.enable_grad():\n                loss = closure()\n        for group in self.param_groups:\n            lr, wd = group["lr"], group["weight_decay"]\n            if group["use_muon"]:\n                _muon_matrix_step(self, group)\n            else:\n                b1, b2 = group["betas"]\n                eps = group["eps"]\n                for p in group["params"]:\n                    if p.grad is None:\n                        continue\n                    g = p.grad\n                    if wd != 0.0:\n                        p.mul_(1.0 - lr * wd)                   # decoupled WD\n                    st = self.state[p]\n                    if len(st) == 0:\n                        st["step"] = 0\n                        st["m"] = torch.zeros_like(p)\n                        st["v"] = torch.zeros_like(p)\n                    st["step"] += 1\n                    t = st["step"]\n                    m, v = st["m"], st["v"]\n                    m.mul_(b1).add_(g, alpha=1.0 - b1)\n                    v.mul_(b2).addcmul_(g, g, value=1.0 - b2)\n                    m_hat = m / (1.0 - b1 ** t)\n                    v_hat = v / (1.0 - b2 ** t)\n                    p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)\n        return loss\n\n\ndef split_params_for_muon(model):\n    """Returns (muon_params, adamw_params). A parameter goes to the Muon side\n    iff it has ndim >= 2 AND its qualified name does not match any entry in\n    model._muon_exclude_names (set by the data fragment for token embeddings /\n    positional embeddings / LM head). Everything else -> AdamW side."""\n    exclude = set(getattr(model, "_muon_exclude_names", []))\n\n    def excluded(name):\n        return any(name == e or name.startswith(e + ".") or name.endswith("." + e)\n                   or ("." + e + ".") in ("." + name + ".") for e in exclude)\n\n    muon_params, adamw_params = [], []\n    for name, p in model.named_parameters():\n        if not p.requires_grad:\n            continue\n        if p.ndim >= 2 and not excluded(name):\n            muon_params.append(p)\n        else:\n            adamw_params.append(p)\n    return muon_params, adamw_params\n\n\n# ---------------------------------------------------------------------------\n#  Shampoo -- per-layer L^{-1/4} G R^{-1/4} preconditioner with AdamW norm\n#  grafting. L = EMA(G G^T), R = EMA(G^T G). Grafting the AdamW update norm\n#  keeps it stable at this scale (ungrafted Shampoo is not). 1-D params\n#  (biases, norms) and 2-D params with a dimension > SHAMPOO_MAXF fall through\n#  to the AdamW graft alone -- the latter guard is NOT in the Kaggle reference\n#  and is added here on purpose: without it a tied 16k-vocab lm_head would\n#  allocate a 16000x16000 preconditioner and eigh it. Decoupled weight decay,\n#  same as every optimizer in this block. `beta` is the second tuned axis\n#  (KNOB_GRID["Shampoo"]); `lr` is the LR-searched value.\n# ---------------------------------------------------------------------------\nclass Shampoo(torch.optim.Optimizer):\n    def __init__(self, params, lr=1e-3, beta=SHAMPOO_BETA, eps=SHAMPOO_EPS,\n                 update_freq=SHAMPOO_UPDATE_FREQ, graft_betas=SHAMPOO_GRAFT_BETAS,\n                 maxf=SHAMPOO_MAXF, weight_decay=0.0):\n        if lr <= 0.0:\n            raise ValueError(f"Shampoo: invalid lr {lr}")\n        if not 0.0 < beta < 1.0:\n            raise ValueError(f"Shampoo: invalid beta {beta}")\n        defaults = dict(lr=lr, beta=beta, eps=eps, update_freq=int(update_freq),\n                        graft_betas=graft_betas, maxf=int(maxf),\n                        weight_decay=weight_decay)\n        super().__init__(params, defaults)\n\n    @staticmethod\n    def _inv_root(M, eps, power=-0.25):\n        L, Q = torch.linalg.eigh(M.double())\n        L = L.clamp_min(eps)\n        return (Q @ torch.diag(L.pow(power)) @ Q.T).to(M.dtype)\n\n    @torch.no_grad()\n    def step(self, closure=None):\n        loss = None\n        if closure is not None:\n            with torch.enable_grad():\n                loss = closure()\n        for group in self.param_groups:\n            b1, b2 = group["graft_betas"]\n            lr, wd, maxf = group["lr"], group["weight_decay"], group["maxf"]\n            for p in group["params"]:\n                if p.grad is None:\n                    continue\n                g = p.grad\n                if g.is_sparse:\n                    raise RuntimeError("Shampoo does not support sparse gradients")\n                st = self.state[p]\n                if "t" not in st:\n                    st["t"] = 0\n                    st["m"] = torch.zeros_like(p)\n                    st["v"] = torch.zeros_like(p)\n                    if g.ndim == 2 and max(g.shape) <= maxf:\n                        st["L"] = torch.zeros(g.size(0), g.size(0), device=g.device)\n                        st["R"] = torch.zeros(g.size(1), g.size(1), device=g.device)\n                st["t"] += 1\n                t = st["t"]\n                if wd != 0.0:\n                    p.mul_(1.0 - lr * wd)                       # decoupled WD\n                st["m"].mul_(b1).add_(g, alpha=1.0 - b1)\n                st["v"].mul_(b2).addcmul_(g, g, value=1.0 - b2)\n                adam_dir = (st["m"] / (1.0 - b1 ** t)) / (\n                    (st["v"] / (1.0 - b2 ** t)).sqrt().add(1e-8))\n                if "L" not in st:                              # 1-D or oversized 2-D\n                    p.add_(adam_dir, alpha=-lr)\n                    continue\n                st["L"].mul_(group["beta"]).add_(g @ g.T, alpha=1.0 - group["beta"])\n                st["R"].mul_(group["beta"]).add_(g.T @ g, alpha=1.0 - group["beta"])\n                if t % group["update_freq"] == 0 or "Li" not in st:\n                    st["Li"] = self._inv_root(st["L"], group["eps"])\n                    st["Ri"] = self._inv_root(st["R"], group["eps"])\n                d = st["Li"] @ g @ st["Ri"]\n                d = d * (adam_dir.norm() / (d.norm() + 1e-12))  # graft the norm\n                p.add_(d, alpha=-lr)\n        return loss\n\n\n# ---------------------------------------------------------------------------\n#  Evie-KF -- Kronecker-factored risk-sensitive preconditioner.\n#\n#      dtheta = -lr * P^-1/2 (I + gamma Sigma_z)^-1/2 P^-1/2 m1hat\n#      P      = diag(sqrt(m2hat))                 (AdamW\'s own curvature proxy)\n#      Sigma_z = P^-1/2 Sigma P^-1/2              (noise covariance, whitened)\n#      Sigma  = Cov(g) from the split batch: delta = (gA - gB)/2, g = (gA + gB)/2\n#      Sigma_z ~= (B kron A)/tr(A) per weight matrix; gain_ij = (1 + gamma mu_i lam_j/tr)^-1/2\n#      Z = U_B [ (U_B^T Y U_A) * gain ] U_A^T     (exact; self-checked against the dense form)\n#\n#  This is a per-parameter (self.state[p]) port of the validated flat-vector\n#  reference in "the reference optimizer". The port is required so the\n#  optimizer state survives state_dict()/resume like every other optimizer in\n#  this harness. Two structural points the port gets right (optimizer specification 4.1):\n#    1. step() cannot compute the split-batch noise itself -- the engine runs the\n#       two half-batch backward passes and hands the result in via set_noise(),\n#       exactly mirroring Sophia\'s set_hessian() pattern. p.grad is (gA+gB)/2.\n#    2. Algorithm step 8 (z <- z * ||y||/||z||) is a norm across ALL parameters,\n#       not per-layer: two passes over the param groups per step() -- accumulate\n#       ||y||^2, ||z||^2 globally, then rescale. This keeps Evie-KF\'s step SIZE\n#       identical to AdamW\'s and isolates the direction change (spec sec 1).\n#\n#  Flags: gamma=0 -> AdamW exactly; diag_only -> Evie-diag; centered=False ->\n#  Evie-KFu; per_layer_norm -> Evie-KFn. Decoupled weight decay, same convention\n#  as the rest of the block.\n# ---------------------------------------------------------------------------\nclass EvieKF(torch.optim.Optimizer):\n    def __init__(self, params, lr=1e-3, betas=EVIEKF_BETAS, eps=EVIEKF_EPS,\n                 weight_decay=0.0, gamma=100.0, beta_sig=EVIEKF_BETA_SIG,\n                 warmup=EVIEKF_WARMUP, refresh=EVIEKF_REFRESH, maxf=EVIEKF_MAXF,\n                 diag_only=False, centered=True, per_layer_norm=False,\n                 gain_clip=EVIEKF_GAIN_CLIP,\n                 diag_relative=True, kron_mean_norm=False, norm_preserve=True,\n                 lagged=False, random_basis=False, telemetry_extra=False):\n        if lr <= 0.0 or gamma < 0.0 or not 0.0 < beta_sig < 1.0:\n            raise ValueError("EvieKF: invalid hyperparameters")\n        defaults = dict(lr=lr, beta1=betas[0], beta2=betas[1], eps=eps,\n                        weight_decay=weight_decay)\n        super().__init__(params, defaults)\n        self.gamma, self.beta_sig = float(gamma), float(beta_sig)\n        self.warmup, self.refresh, self.maxf = int(warmup), int(refresh), int(maxf)\n        self.diag_only, self.centered = bool(diag_only), bool(centered)\n        self.per_layer_norm, self.gain_clip = bool(per_layer_norm), float(gain_clip)\n        # --- three diagnostic switches. Every default reproduces the shipped\n        # behaviour EXACTLY, so no protocol run is affected by their existence.\n        #   diag_relative   False -> the diagonal gate is (1 + gamma*s)^-1/2, the\n        #                   derivation\'s own diagonal special case, instead of the\n        #                   mean-normalised (1 + gamma*s/mean(s))^-1/2. Needed to\n        #                   deconfound C2 (DERIVATION.md sec 5.1, FW-11).\n        #   kron_mean_norm  True -> normalise the Kronecker spectrum by its MEAN\n        #                   eigenvalue instead of tr(A), which is the only form in\n        #                   which gamma is actually dimensionless (FW-9).\n        #   norm_preserve   False -> skip the global ||y||/||z|| rescale, so the\n        #                   operator shrinks instead of rotating. Tests whether\n        #                   norm preservation is load-bearing (FW-10).\n        self.lagged, self.random_basis = bool(lagged), bool(random_basis)\n        self.telemetry_extra = bool(telemetry_extra)\n        self._param_indices = {id(p): i for i, p in enumerate(\n            p for group in self.param_groups for p in group["params"])}\n        self._extra = dict(c_sum=0., c_n=0, c_min=float("inf"), eig_s=0.)\n        if self.telemetry_extra:\n            _counts=[0,0,0]\n            for _group in self.param_groups:\n                if _group.get("use_muon",False):continue\n                for _p in _group["params"]:\n                    _branch=2 if _p.ndim<2 or diag_only else (0 if _p.size(0)<=self.maxf and _p.numel()//_p.size(0)<=self.maxf else 1)\n                    _counts[_branch]+=_p.numel()\n            for _key,_value in zip(("frac_params_kron","frac_params_diag_fallback","frac_params_vector"),_counts):\n                setattr(self,_key,_value/max(1,sum(_counts)))\n        self.diag_relative = bool(diag_relative)\n        self.kron_mean_norm = bool(kron_mean_norm)\n        self.norm_preserve = bool(norm_preserve)\n        self._roles = {}                # {param: role_str}; set by set_param_roles()\n        # descriptive diagnostics, refreshed every step(), read by the engine\n        # (optimizer specification sec 4.3.4/4.3.5). NEVER significance-tested.\n        self.last_cos = 1.0\n        self.b_simple = 0.0\n        self.tr_sigma = 0.0\n        self.eff_rank = 0.0\n        self.gain_lo, self.gain_hi = 1.0, 1.0\n        self.n_active_steps = 0        # wiring canary: 0 at end of run => set_noise()\n                                      # was never called and this WAS silently AdamW\n        self.role_gain = {}           # role -> [gain_lo, gain_hi, eff_rank_sum, n]\n\n    def state_dict(self):\n        value = super().state_dict()\n        if self.telemetry_extra:\n            value["extra_telemetry"] = dict(self._extra)\n        return value\n\n    def load_state_dict(self, value):\n        value = dict(value)\n        extra = value.pop("extra_telemetry", None)\n        super().load_state_dict(value)\n        if extra is not None:\n            self._extra = dict(extra)\n\n    def set_param_roles(self, param_roles):\n        """{param: role} with role in {embedding, attention, ffn, head, other};\n        enables the per-module-category gain/effective-rank breakdown (figures\n        F4/F5). Optional -- without it role_gain stays empty."""\n        self._roles = dict(param_roles)\n\n    @staticmethod\n    def _eigh(M):\n        Ms = (M + M.transpose(-1, -2)) * 0.5\n        L, W = torch.linalg.eigh(Ms.double())          # fp64 eigh regardless of\n        return L.clamp_min(0).to(M.dtype), W.to(M.dtype)   # param dtype, then cast back\n\n    @torch.no_grad()\n    def set_noise(self, param_delta_pairs):\n        """Called by the engine once per step BEFORE step(), with\n        param_delta_pairs = [(p, (gradA - gradB)/2), ...] and p.grad already set\n        to (gradA + gradB)/2. Mirrors Sophia.set_hessian()."""\n        for p, delta in param_delta_pairs:\n            self.state[p]["_delta"] = delta.detach()\n\n    def _global_step(self):\n        """The global step index, derived from the per-parameter counters so it\n        survives load_state_dict() (self.__dict__ is not checkpointed). Equals a\n        plain monotone counter as long as every param is stepped together."""\n        s = 0\n        for group in self.param_groups:\n            for p in group["params"]:\n                st = self.state.get(p)\n                if st and "step" in st:\n                    s = max(s, int(st["step"]))\n        return s + 1\n\n    @torch.no_grad()\n    def step(self, closure=None):\n        loss = None\n        if closure is not None:\n            with torch.enable_grad():\n                loss = closure()\n        t = self._global_step()\n        active_gate = (t > self.warmup) and (self.gamma > 0.0)\n\n        cache = []                       # (p, group, y, z, pm12) for pass 2\n        y_sq_total = z_sq_total = dot_total = 0.0\n        tr_sigma_total = y_sq_for_bsimple = 0.0\n        gains_lo, gains_hi, eff_ranks = [], [], []\n        any_noise = False\n        self.role_gain = {}\n\n        if self.telemetry_extra:\n            _tele = {}\n        # ---- pass 1: per-parameter y (whitened Adam dir) and z (preconditioned) --\n        for group in self.param_groups:\n            if group.get("use_muon", False):\n                continue\n            b1, b2, eps = group["beta1"], group["beta2"], group["eps"]\n            for p in group["params"]:\n                if p.grad is None:\n                    continue\n                g = p.grad\n                st = self.state[p]\n                if "step" not in st:\n                    st["step"] = 0\n                    st["m"] = torch.zeros_like(p)\n                    st["v"] = torch.zeros_like(p)\n                st["step"] += 1\n                pt = st["step"]\n                st["m"].mul_(b1).add_(g, alpha=1.0 - b1)\n                st["v"].mul_(b2).addcmul_(g, g, value=1.0 - b2)\n                mh = st["m"] / (1.0 - b1 ** pt)\n                vh = st["v"] / (1.0 - b2 ** pt)\n                pm12 = vh.sqrt().add(eps).rsqrt()          # == torch AdamW\'s P^-1/2\n                y = pm12 * mh\n                z = y.clone()\n\n                delta = st.pop("_delta", None)             # consume; never stale-reuse\n                if delta is not None:\n                    any_noise = True\n                    src = delta if self.centered else g\n                    dz = pm12 * src.detach().float()\n                    tr_sigma_total += float((dz * dz).sum())\n                    y_sq_for_bsimple += float((y * y).sum())\n\n                    if p.dim() < 2 or self.diag_only:\n                        if "s" not in st:\n                            st["s"] = torch.zeros_like(dz)\n                        if self.lagged:\n                            _old_s = st["s"].clone()\n                        st["s"].mul_(self.beta_sig).addcmul_(\n                            dz, dz, value=1.0 - self.beta_sig)\n                        if active_gate:\n                            s = st["s"] / (1.0 - self.beta_sig ** pt)\n                            if self.lagged:\n                                s = _old_s / (1.0 - self.beta_sig ** (pt-1))\n                            gsc = (1.0 + self.gamma\n                                   * (s / (s.mean() + 1e-30)\n                                      if self.diag_relative else s)).rsqrt()\n                            if self.per_layer_norm:\n                                gsc = (gsc / gsc.pow(2).mean().sqrt()\n                                       .clamp_min(1e-30)).clamp(max=self.gain_clip)\n                            z = y * gsc\n                    else:\n                        d0 = p.size(0)\n                        rest = p.numel() // d0\n                        D = dz.reshape(d0, rest)\n                        if "A" not in st:\n                            st["A"] = (torch.zeros(rest, rest, device=p.device)\n                                       if rest <= self.maxf\n                                       else torch.zeros(rest, device=p.device))\n                            st["B"] = (torch.zeros(d0, d0, device=p.device)\n                                       if d0 <= self.maxf\n                                       else torch.zeros(d0, device=p.device))\n                        if self.lagged:\n                            _old_A, _old_B = st["A"].clone(), st["B"].clone()\n                        st["A"].mul_(self.beta_sig).add_(\n                            (D.T @ D) if rest <= self.maxf else (D * D).sum(0),\n                            alpha=1.0 - self.beta_sig)\n                        st["B"].mul_(self.beta_sig).add_(\n                            (D @ D.T) if d0 <= self.maxf else (D * D).sum(1),\n                            alpha=1.0 - self.beta_sig)\n                        if active_gate:\n                            if (pt % self.refresh == 0) or ("lam" not in st):\n                                A = st["A"] / (1.0 - self.beta_sig ** pt)\n                                B = st["B"] / (1.0 - self.beta_sig ** pt)\n                                if self.lagged:\n                                    A = _old_A / (1.0 - self.beta_sig ** (pt-1))\n                                    B = _old_B / (1.0 - self.beta_sig ** (pt-1))\n                                if self.telemetry_extra:\n                                    if p.is_cuda:\n                                        torch.cuda.synchronize()\n                                    import time as _time\n                                    _eig_start = _time.perf_counter()\n                                st["lam"], st["UA"] = (\n                                    self._eigh(A) if A.dim() == 2\n                                    else (A.clamp_min(0), None))\n                                st["mu"], st["UB"] = (\n                                    self._eigh(B) if B.dim() == 2\n                                    else (B.clamp_min(0), None))\n                                if self.telemetry_extra:\n                                    if p.is_cuda:\n                                        torch.cuda.synchronize()\n                                    self._extra["eig_s"] += _time.perf_counter() - _eig_start\n                                    for _side, _F, _ev in [("A", A, st["lam"]), ("B", B, st["mu"])]:\n                                        _den = float(_F.square().sum())\n                                        _off = (max(0., _den-float(_F.diagonal().square().sum()))/_den\n                                                if _F.ndim == 2 and _den > 0 else 0.)\n                                        _tele.setdefault("offdiag_frac_"+_side,[]).append(_off)\n                                        _tele.setdefault("anisotropy_"+_side,[]).append(float(_ev.max()/(_ev.mean()+1e-30)))\n                                if self.random_basis:\n                                    # Kronecker-only control: vector gates and diagonal\n                                    # fallback sides retain their original directions.\n                                    for _side, _cache, _number in [("UA", "RA", 0), ("UB", "RB", 1)]:\n                                        if st[_side] is not None:\n                                            if _cache not in st:\n                                                _gen = torch.Generator(device="cpu").manual_seed(\n                                                    1000003*self._param_indices[id(p)]+_number)\n                                                _n = st[_side].size(0)\n                                                _Q, _R = torch.linalg.qr(torch.randn(_n, _n, generator=_gen, dtype=torch.float32))\n                                                st[_cache] = (_Q*torch.sign(torch.diagonal(_R))).to(p)\n                                            st[_side] = st[_cache]\n                                _tr_a = float(st["lam"].sum())\n                                # /tau matches tr(B kron A) to tr(Sigma_z); it is\n                                # NOT scale-free. /mean(mu*lam) is (FW-9).\n                                st["tr"] = ((_tr_a * float(st["mu"].sum())\n                                             / max(1, p.numel()))\n                                            if self.kron_mean_norm\n                                            else _tr_a) + 1e-30\n                            lam, mu = st["lam"], st["mu"]\n                            UA, UB, tr = st["UA"], st["UB"], st["tr"]\n                            C = y.reshape(d0, rest)\n                            if UB is not None:\n                                C = UB.T @ C\n                            if UA is not None:\n                                C = C @ UA\n                            gain = (1.0 + self.gamma\n                                    * (mu[:, None] * lam[None, :]) / tr).rsqrt()\n                            if self.per_layer_norm:\n                                gain = (gain / gain.pow(2).mean().sqrt()\n                                        .clamp_min(1e-30)).clamp(max=self.gain_clip)\n                            g_lo, g_hi = float(gain.min()), float(gain.max())\n                            ev = (mu[:, None] * lam[None, :]).reshape(-1)\n                            er = float(ev.sum() ** 2 / (ev.pow(2).sum() + 1e-30))\n                            gains_lo.append(g_lo)\n                            gains_hi.append(g_hi)\n                            eff_ranks.append(er)\n                            role = self._roles.get(p, "other")\n                            acc_r = self.role_gain.setdefault(\n                                role, [g_lo, g_hi, 0.0, 0])\n                            acc_r[0] = min(acc_r[0], g_lo)\n                            acc_r[1] = max(acc_r[1], g_hi)\n                            acc_r[2] += er\n                            acc_r[3] += 1\n                            if self.telemetry_extra and ((pt % self.refresh == 0) or pt == self.warmup+1):\n                                _spectrum = (mu[:,None]*lam[None,:]).reshape(-1)\n                                _idx = _spectrum.topk(max(1, math.ceil(.1*_spectrum.numel()))).indices\n                                _before = C.reshape(-1).square()\n                                _after = (C*gain).reshape(-1).square()\n                                _tele.setdefault("energy_top_before",[]).append(float(_before[_idx].sum()/(_before.sum()+1e-30)))\n                                _tele.setdefault("energy_top_after",[]).append(float(_after[_idx].sum()/(_after.sum()+1e-30)))\n                            C = C * gain\n                            if UB is not None:\n                                C = UB @ C\n                            if UA is not None:\n                                C = C @ UA.T\n                            z = C.reshape_as(p)\n\n                y_sq_total += float((y * y).sum())\n                z_sq_total += float((z * z).sum())\n                dot_total += float((y * z).sum())\n                cache.append((p, group, y, z, pm12))\n\n        # ---- pass 2: global norm-preserving rescale, then apply the update ------\n        ny = y_sq_total ** 0.5\n        nz = z_sq_total ** 0.5\n        # `op_active` is "the operator ran this step" and drives the telemetry and\n        # the wiring canary; `do_rescale` is the separate question of whether the\n        # global norm rescale is applied. They were one variable until EvieKFr\n        # needed to disable the rescale WITHOUT making n_active_steps read 0,\n        # which the engine treats as a silent-AdamW bug and flags as a divergence.\n        op_active = active_gate and any_noise and nz > 1e-30\n        do_rescale = op_active and self.norm_preserve\n        scale = (ny / nz) if do_rescale else 1.0\n        if self.telemetry_extra:\n            if op_active:\n                self._extra["c_sum"] += scale\n                self._extra["c_n"] += 1\n                self._extra["c_min"] = min(self._extra["c_min"], scale)\n            self.last_c_mean = self._extra["c_sum"]/max(1,self._extra["c_n"])\n            self.last_c_min = self._extra["c_min"] if self._extra["c_n"] else float("nan")\n            self.eig_refresh_time_s = self._extra["eig_s"]\n            for _key, _values in _tele.items():\n                if _values:\n                    setattr(self, _key, sum(_values)/len(_values))\n        den = ny * nz\n        self.last_cos = (dot_total / den) if (op_active and den > 1e-30) else 1.0\n        self.tr_sigma = tr_sigma_total\n        self.b_simple = (tr_sigma_total / (y_sq_for_bsimple + 1e-30)\n                         if any_noise else 0.0)\n        if gains_lo:\n            self.gain_lo, self.gain_hi = min(gains_lo), max(gains_hi)\n            self.eff_rank = sum(eff_ranks) / len(eff_ranks)\n        if op_active:\n            self.n_active_steps += 1\n\n        for p, group, y, z, pm12 in cache:\n            lr, wd = group["lr"], group["weight_decay"]\n            if wd != 0.0:\n                p.mul_(1.0 - lr * wd)                          # decoupled WD\n            u = (pm12 * (z * scale)) if scale != 1.0 else (pm12 * z)\n            p.add_(u.to(p.dtype), alpha=-lr)\n        return loss\n\n\ndef eviekf_self_check():\n    """Dense-vs-factored (B kron A)^-1/2 parity + the gamma=0 identity\n    (optimizer specification sec 4.3.3 / sec 5). Ported from kaggle_evie_v3.py\'s\n    self_check(). Raises RuntimeError on failure -- the engine turns that into a\n    one-sentence abort, exactly like the CUDA preflight. A vec-ordering / eigh\n    bug here produces a plausible-looking but meaningless results table."""\n    gen = torch.Generator().manual_seed(0)\n    n, m, gam = 5, 4, 3.7\n    D = torch.randn(40, n, m, generator=gen, dtype=torch.float64)\n    A = torch.einsum("bij,bik->jk", D, D) / 40.0\n    B = torch.einsum("bij,bkj->ik", D, D) / 40.0\n    tr = float(A.trace())\n    Y = torch.randn(n, m, generator=gen, dtype=torch.float64)\n    lam, UA = torch.linalg.eigh(A)\n    mu, UB = torch.linalg.eigh(B)\n    gain = (1.0 + gam * (mu[:, None] * lam[None, :]) / tr).rsqrt()\n    fast = UB @ ((UB.T @ Y @ UA) * gain) @ UA.T\n    M = torch.eye(n * m, dtype=torch.float64) + gam * torch.kron(B, A) / tr\n    w, V = torch.linalg.eigh(M)\n    dense = (V @ torch.diag(w.rsqrt()) @ V.T) @ Y.reshape(-1)\n    e1 = float((fast.reshape(-1) - dense).abs().max())\n    g1 = (1.0 + 0.0 * (mu[:, None] * lam[None, :]) / tr).rsqrt()\n    e2 = float((UB @ ((UB.T @ Y @ UA) * g1) @ UA.T - Y).abs().max())\n    ok3 = bool((gain <= 1.0 + 1e-12).all())\n    if not (e1 < 1e-9 and e2 < 1e-9 and ok3):\n        raise RuntimeError(\n            f"EvieKF Kronecker self-check FAILED (factored-vs-dense max err "\n            f"{e1:.2e}, gamma=0 identity err {e2:.2e}, gains<=1 {ok3}); refusing "\n            f"to run -- every Evie-KF number would be meaningless.")\n    return {"factored_vs_dense": e1, "gamma0_identity": e2, "gains_le_1": ok3}\n\n\nclass MuonEvie(EvieKF):\n    """Muon matrices plus EvieKFm auxiliaries; norm preservation and all\n    inherited mechanism diagnostics cover ONLY the Evie parameter group."""\n    def __init__(self, muon_params, evie_params, matrix_lr, sub_lr, gamma,\n                 **evie_kwargs):\n        evie_params = list(evie_params)\n        self._evie_ids = {id(p) for p in evie_params}\n        groups = [dict(params=list(muon_params), lr=matrix_lr, use_muon=True,\n                       momentum=MUON_MOMENTUM, nesterov=MUON_NESTEROV,\n                       ns_steps=MUON_NS_STEPS, weight_decay=WEIGHT_DECAY),\n                  dict(params=evie_params, lr=sub_lr, use_muon=False)]\n        super().__init__(groups, lr=sub_lr, gamma=gamma, **evie_kwargs)\n\n    @torch.no_grad()\n    def set_noise(self, pairs):\n        super().set_noise((p, d) for p, d in pairs if id(p) in self._evie_ids)\n\n    @torch.no_grad()\n    def step(self, closure=None):\n        loss = None\n        if closure is not None:\n            with torch.enable_grad():\n                loss = closure()\n        for group in self.param_groups:\n            if group.get("use_muon", False):\n                _muon_matrix_step(self, group)\n        super().step()\n        return loss\n\n    def state_dict(self):\n        state = super().state_dict()\n        # EvieKF derives arithmetic counters from per-parameter state. Its\n        # diagnostic activity counter must also survive a mid-run resume.\n        state["muonevie_active_steps"] = self.n_active_steps\n        return state\n\n    def load_state_dict(self, state_dict):\n        state_dict = dict(state_dict)\n        active = state_dict.pop("muonevie_active_steps", 0)\n        super().load_state_dict(state_dict)\n        self.n_active_steps = int(active)\n        self._evie_ids = {id(p) for g in self.param_groups\n                          if not g.get("use_muon", False) for p in g["params"]}\n\n\ndef muonevie_self_check():\n    """CPU float32 nesting, gate isolation, tied split, and resume canaries."""\n    import copy\n    class Tiny(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.embedding = nn.Parameter(torch.zeros(7, 4))\n            self.bias = nn.Parameter(torch.zeros(4))\n            self.hidden1 = nn.Parameter(torch.zeros(4, 4))\n            self.hidden2 = nn.Parameter(torch.zeros(4, 5))\n            self._muon_exclude_names = {"embedding"}\n    gen = torch.Generator().manual_seed(1907)\n    def feed(models, opts):\n        for ps in zip(*(m.parameters() for m in models)):\n            grad = torch.randn(ps[0].shape, generator=gen)\n            delta = torch.randn(ps[0].shape, generator=gen)\n            for p, opt in zip(ps, opts):\n                p.grad = grad.clone()\n                if hasattr(opt, "set_noise"):\n                    opt.set_noise([(p, delta.clone())])\n        for opt in opts:\n            opt.step()\n    def gap(a, b, evie=None):\n        return max(float((p-q).detach().abs().max()) for (name,p),q in\n                   zip(a.named_parameters(), b.parameters())\n                   if evie is None or ((name in {"embedding", "bias"}) == evie))\n    for gamma, steps in [(0, 8), (300, 40)]:\n        a, b = Tiny(), Tiny()\n        oa = build_optimizer("MuonSub", a, .001, matrix_lr=.002,\n                             knob=WEIGHT_DECAY)\n        ob = build_optimizer("MuonEvie", b, .001, matrix_lr=.002, knob=gamma)\n        for _ in range(steps):\n            feed([a,b], [oa,ob])\n        if gamma == 0 and gap(a,b) > 1e-6:\n            raise RuntimeError("MuonEvie gamma=0 does not nest MuonSub.")\n        if gamma and (gap(a,b,True) <= 1e-5 or gap(a,b,False) > 1e-6\n                      or ob.n_active_steps <= 0):\n            raise RuntimeError("MuonEvie gate is inactive or modifies Muon matrices.")\n        c = copy.deepcopy(b)\n        oc = build_optimizer("MuonEvie", c, .001, matrix_lr=.002, knob=gamma)\n        oc.load_state_dict(copy.deepcopy(ob.state_dict()))\n        feed([b,c], [ob,oc])\n        if gap(b,c) > 1e-6 or ob.n_active_steps != oc.n_active_steps:\n            raise RuntimeError("MuonEvie checkpoint resume does not round-trip.")\n    return True\n\n\ndef evie_controls_self_check():\n    """Gamma-zero, genuinely lagged gate, Kronecker-only random basis,\n    telemetry arithmetic/RNG parity, and control checkpoint round-trip."""\n    import copy\n    gen = torch.Generator().manual_seed(841)\n    def make(**kw):\n        p = nn.Parameter(torch.ones(4, 5, dtype=torch.float32))\n        return p, EvieKF([p], lr=.01, weight_decay=WEIGHT_DECAY,\n                         refresh=10, **kw)\n    def update(p, opt, grad, delta):\n        p.grad = grad.clone()\n        if hasattr(opt, "set_noise"):\n            opt.set_noise([(p, delta.clone())])\n        opt.step()\n    for flag in ["lagged", "random_basis"]:\n        p, opt = make(gamma=0., **{flag:True})\n        q = nn.Parameter(p.detach().clone())\n        ref = torch.optim.AdamW([q], lr=.01, betas=EVIEKF_BETAS,\n                               eps=EVIEKF_EPS, weight_decay=WEIGHT_DECAY)\n        for _ in range(5):\n            g = torch.randn(p.shape, generator=gen)\n            d = torch.randn(p.shape, generator=gen)\n            update(p,opt,g,d);update(q,ref,g,d)\n        if float((p-q).detach().abs().max()) > 1e-6:\n            raise RuntimeError("Evie control gamma-zero nesting failed")\n    for lagged in [True,False]:\n        p, opt = make(gamma=300., lagged=lagged)\n        for _ in range(39):\n            update(p,opt,torch.randn(p.shape,generator=gen),torch.randn(p.shape,generator=gen))\n        q, other = make(gamma=300.,lagged=lagged)\n        q.data.copy_(p.data);other.load_state_dict(copy.deepcopy(opt.state_dict()))\n        grad = torch.randn(p.shape,generator=gen)\n        d = torch.randn(p.shape,generator=gen)\n        update(p,opt,grad,d);update(q,other,grad,50*d.flip(0))\n        gap = float((p-q).detach().abs().max())\n        if (lagged and gap >= 1e-7) or (not lagged and gap <= 1e-6):\n            raise RuntimeError("Lagged gate canary failed to isolate current noise")\n    p, opt = make(gamma=30., random_basis=True)\n    q, ref = make(gamma=30.)\n    a, logged = make(gamma=30., telemetry_extra=True)\n    global_rng = torch.random.get_rng_state().clone()\n    for _ in range(30):\n        g,d = torch.randn(p.shape,generator=gen),torch.randn(p.shape,generator=gen)\n        update(p,opt,g,d);update(q,ref,g,d);update(a,logged,g,d)\n    if not torch.equal(global_rng,torch.random.get_rng_state()) or not torch.equal(q,a):\n        raise RuntimeError("Extra Evie telemetry changed arithmetic or global RNG")\n    st, sr = opt.state[p],ref.state[q]\n    RA = st["RA"]\n    if not torch.allclose(RA.T@RA,torch.eye(RA.size(0)),atol=1e-5):\n        raise RuntimeError("Random-basis control is not orthogonal")\n    # Noise whitening is gradient-only, hence shared spectra despite changed weights.\n    ga=(1+30*st["mu"][:,None]*st["lam"][None,:]/st["tr"]).rsqrt()\n    gb=(1+30*sr["mu"][:,None]*sr["lam"][None,:]/sr["tr"]).rsqrt()\n    if not torch.equal(ga,gb):\n        raise RuntimeError("Random-basis control changed the gate spectrum")\n    q, restored=make(gamma=30.,random_basis=True)\n    q.data.copy_(p);restored.load_state_dict(copy.deepcopy(opt.state_dict()))\n    g,d=torch.randn(p.shape,generator=gen),torch.randn(p.shape,generator=gen)\n    update(p,opt,g,d);update(q,restored,g,d)\n    if not torch.equal(p,q):\n        raise RuntimeError("Evie control checkpoint round-trip failed")\n    return True\n\n\ndef build_optimizer(opt_name, model, lr, adamw_lr=None, knob=None,\n                    eviekf_maxf=None, matrix_lr=None, telemetry_extra=False):\n    """Single optimizer-selection entry point (benchmark protocol sec 1: keep this\n    structured so a new optimizer pastes in as one extra branch). `lr` is the\n    frozen, LR-searched value for `opt_name`. `adamw_lr` is only used by Muon\n    (its AdamW sub-group). `knob` is the frozen value of the SECOND tuned axis\n    for optimizers in KNOB_GRID -- gamma for the Evie-KF family, beta for\n    Shampoo -- and is ignored by every other optimizer. Weight decay is\n    WEIGHT_DECAY (decoupled) for every optimizer.\n\n    `eviekf_maxf` overrides the frozen EVIEKF_MAXF Kronecker size limit and is\n    passed by the engine ONLY when the task declares TASK["eviekf_maxf"] -- a\n    diagnostic file, never a protocol task (see the note on EVIEKF_MAXF). It\n    arrives as an argument rather than being read from TASK here so that this\n    block stays free of any dependency on the task header."""\n    params = [p for p in model.parameters() if p.requires_grad]\n    if opt_name.endswith("wd") and opt_name in ("AdamWwd", "AdaBeliefwd", "SGDwd", "Lionwd", "Muonwd", "EvieKFmwd"):\n        wd = WEIGHT_DECAY if knob is None else float(knob)\n        if not math.isfinite(wd) or wd < 0:\n            raise ValueError("Weight decay must be finite and nonnegative")\n        if opt_name == "EvieKFmwd":\n            opt = build_optimizer("EvieKFm", model, lr, knob=EVIEKF_FIXED_GAMMA_M,\n                                  telemetry_extra=telemetry_extra)\n        else:\n            opt = build_optimizer(opt_name[:-2], model, lr, adamw_lr=adamw_lr,\n                                  telemetry_extra=telemetry_extra)\n        for group in opt.param_groups:\n            group["weight_decay"] = wd\n        return opt\n    if opt_name in ("MuonSub", "MuonEvie", "MuonEvieFix"):\n        if matrix_lr is None or not math.isfinite(matrix_lr) or matrix_lr <= 0:\n            raise ValueError("Muon hybrid requires a positive frozen matrix_lr")\n        muon_params, rest = split_params_for_muon(model)\n        if not muon_params or not rest:\n            raise RuntimeError("Muon hybrid needs trainable matrix and auxiliary parameters")\n        if opt_name == "MuonSub":\n            return Muon([dict(params=muon_params, lr=matrix_lr, use_muon=True),\n                         dict(params=rest, lr=lr, use_muon=False,\n                              weight_decay=WEIGHT_DECAY if knob is None else float(knob))])\n        return MuonEvie(muon_params, rest, matrix_lr, lr,\n                        gamma=30.0 if knob is None else float(knob),\n                        betas=EVIEKF_BETAS, eps=EVIEKF_EPS,\n                        weight_decay=WEIGHT_DECAY, beta_sig=EVIEKF_BETA_SIG,\n                        warmup=EVIEKF_WARMUP, refresh=EVIEKF_REFRESH,\n                        maxf=EVIEKF_MAXF, diag_only=False, centered=True,\n                        per_layer_norm=False, gain_clip=EVIEKF_GAIN_CLIP,\n                        diag_relative=True, kron_mean_norm=True, norm_preserve=True,\n                        telemetry_extra=telemetry_extra)\n    if opt_name == "AdamW":\n        return torch.optim.AdamW(params, lr=lr, betas=(0.9, 0.999), eps=1e-8,\n                                 weight_decay=WEIGHT_DECAY)\n    if opt_name == "AdaBelief":\n        return AdaBelief(params, lr=lr, betas=ADABELIEF_BETAS, eps=ADABELIEF_EPS,\n                         weight_decay=WEIGHT_DECAY)\n    if opt_name == "SGD":\n        return SGDM(params, lr=lr, momentum=SGDM_MOMENTUM, nesterov=SGDM_NESTEROV,\n                    weight_decay=WEIGHT_DECAY)\n    if opt_name == "Sophia":\n        return Sophia(params, lr=lr, betas=SOPHIA_BETAS, rho=SOPHIA_RHO,\n                      eps=SOPHIA_EPS, weight_decay=WEIGHT_DECAY)\n    if opt_name == "Lion":\n        return Lion(params, lr=lr, betas=LION_BETAS, weight_decay=WEIGHT_DECAY)\n    if opt_name == "Shampoo":\n        return Shampoo(params, lr=lr,\n                       beta=(float(knob) if knob is not None else SHAMPOO_BETA),\n                       eps=SHAMPOO_EPS, update_freq=SHAMPOO_UPDATE_FREQ,\n                       graft_betas=SHAMPOO_GRAFT_BETAS, maxf=SHAMPOO_MAXF,\n                       weight_decay=WEIGHT_DECAY)\n    if opt_name in ("EvieKFn", "EvieKF", "EvieKFu", "EvieDiag",\n                    "EvieDiagAbs", "EvieKFm", "EvieKFr", "EvieKFlag", "EvieKFrb", "EvieKFmrb"):\n        return EvieKF(params, lr=lr, betas=EVIEKF_BETAS, eps=EVIEKF_EPS,\n                      weight_decay=WEIGHT_DECAY,\n                      gamma=(float(knob) if knob is not None else 100.0),\n                      beta_sig=EVIEKF_BETA_SIG, warmup=EVIEKF_WARMUP,\n                      refresh=EVIEKF_REFRESH,\n                      maxf=(EVIEKF_MAXF if eviekf_maxf is None\n                            else int(eviekf_maxf)),\n                      diag_only=(opt_name in ("EvieDiag", "EvieDiagAbs")),\n                      centered=(opt_name != "EvieKFu"),\n                      per_layer_norm=(opt_name == "EvieKFn"),\n                      gain_clip=EVIEKF_GAIN_CLIP,\n                      diag_relative=(opt_name != "EvieDiagAbs"),\n                      kron_mean_norm=(opt_name in ("EvieKFm", "EvieKFmrb")),\n                      norm_preserve=(opt_name != "EvieKFr"),\n                      lagged=(opt_name == "EvieKFlag"), random_basis=(opt_name in ("EvieKFrb", "EvieKFmrb")),\n                      telemetry_extra=telemetry_extra)\n    if opt_name == "Muon":\n        muon_params, adamw_params = split_params_for_muon(model)\n        a_lr = adamw_lr if adamw_lr is not None else lr\n        groups = []\n        if muon_params:\n            groups.append(dict(params=muon_params, lr=lr, use_muon=True))\n        if adamw_params:\n            groups.append(dict(params=adamw_params, lr=a_lr, use_muon=False))\n        if not groups:\n            raise RuntimeError("Muon: model has no trainable parameters")\n        return Muon(groups)\n    raise ValueError(f"unknown optimizer {opt_name!r}")\n\n\ndef optimizer_lr_scale(optimizer):\n    """Base LRs per param-group, so the engine\'s cosine scheduler can be rebuilt\n    on resume with the right per-group base_lrs (Muon has two)."""\n    return [g["lr"] for g in optimizer.param_groups]\n# === END SHARED OPTIMIZER BLOCK ===\n', encoding='utf-8')
Path('pilot.py').write_text('"""Compare benchmark optimizers on a matrix quadratic with rotated noise."""\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport time\nfrom pathlib import Path\nfrom statistics import median\n\nimport numpy as np\nimport torch\n\nfrom pilot_optimizers import EvieKF, Muon\n\n\nARMS = ("AdamW", "Muon", "Evie-KF", "Evie diagonal", "Evie random basis")\n\n\ndef rotated_factor(spectrum, fraction):\n    """Rotate opposite ends of the spectrum through disjoint planes."""\n    dim = len(spectrum)\n    basis = np.eye(dim)\n    angle = fraction * math.pi / 4\n    c, s = math.cos(angle), math.sin(angle)\n    for i in range(dim // 2):\n        pair = [i, dim - 1 - i]\n        basis[np.ix_(pair, pair)] = [[c, -s], [s, c]]\n    return basis * np.sqrt(spectrum)\n\n\ndef covariance_inputs(data, fraction):\n    left = rotated_factor(data["row_eigenvalues"], fraction)\n    right = rotated_factor(data["column_eigenvalues"], fraction)\n    covariance = np.kron(left @ left.T, right @ right.T)\n    expected = np.sort(np.outer(data["row_eigenvalues"], data["column_eigenvalues"]).ravel())\n    np.testing.assert_allclose(np.linalg.eigvalsh(covariance), expected, rtol=1e-9, atol=1e-11)\n    off_diagonal = covariance - np.diag(np.diag(covariance))\n    return left, right, float(np.linalg.norm(off_diagonal) / np.linalg.norm(covariance))\n\n\ndef problem(data, fraction, seed, steps, device):\n    left, right, _ = covariance_inputs(data, fraction)\n    rng = np.random.Generator(np.random.PCG64(seed))\n    shape = (left.shape[0], right.shape[0])\n    initial = rng.standard_normal(shape)\n    normals = rng.standard_normal((steps, 2, *shape))\n    # Each half has twice the target covariance, so their average has it.\n    halves = math.sqrt(2) * (left @ normals @ right.T)\n    curvature = np.asarray(data["curvature"])\n    return tuple(torch.as_tensor(x, device=device, dtype=torch.float32)\n                 for x in (initial, halves[:, 0], halves[:, 1], curvature))\n\n\ndef optimizer(arm, param, lr, gamma):\n    if arm == "AdamW":\n        return torch.optim.AdamW([param], lr=lr, weight_decay=0)\n    if arm == "Muon":\n        return Muon([dict(params=[param], lr=lr, use_muon=True, weight_decay=0)])\n    return EvieKF([param], lr=lr, gamma=gamma, weight_decay=0, kron_mean_norm=True,\n                  diag_only=arm == "Evie diagonal", random_basis=arm == "Evie random basis")\n\n\ndef sync(device):\n    if device.type == "cuda":\n        torch.cuda.synchronize(device)\n\n\n@torch.no_grad()\ndef rollout(arm, inputs, lr, gamma, every, timed):\n    initial, noise_a, noise_b, curvature = inputs\n    weight = torch.nn.Parameter(initial.clone())\n    opt = optimizer(arm, weight, lr, gamma)\n    evie = isinstance(opt, EvieKF)\n    loss = lambda: float((curvature * weight.square()).sum() * 0.5)\n    trace = [[0, 0.0, loss()]]\n    elapsed = 0.0\n    sync(weight.device)\n    start = time.perf_counter()\n    steps = len(noise_a)\n    for step in range(1, steps + 1):\n        true_grad = curvature * weight\n        a, b = true_grad + noise_a[step - 1], true_grad + noise_b[step - 1]\n        weight.grad = (a + b) * 0.5\n        if evie:\n            opt.set_noise([(weight, (a - b) * 0.5)])\n        opt.step()\n        if step == steps or (timed and step % every == 0):\n            sync(weight.device)\n            elapsed += time.perf_counter() - start\n            value = loss()\n            if not math.isfinite(value):\n                return None\n            trace.append([step, elapsed, value])\n            start = time.perf_counter()\n    if evie and steps > opt.warmup and opt.n_active_steps == 0:\n        raise RuntimeError(f"{arm} never activated the gate")\n    return trace\n\n\ndef run(data, out, device, smoke=False):\n    full_steps = data["steps"]\n    steps = 60 if smoke else full_steps\n    tune_seeds = data["tune_seeds"][:1] if smoke else data["tune_seeds"]\n    eval_seeds = data["evaluation_seeds"][:2] if smoke else data["evaluation_seeds"]\n    fractions = data["rotations"]\n    gamma = data["gamma"]\n    tuning, trials, summary = [], [], []\n    assert not set(tune_seeds) & set(eval_seeds)\n    out.mkdir(parents=True, exist_ok=True)\n\n    for fraction in fractions:\n        _, _, off_diagonal = covariance_inputs(data, fraction)\n        lr_by_arm = {}\n        tune_inputs = [problem(data, fraction, seed, steps, device) for seed in tune_seeds]\n        for arm in ARMS:\n            grid = data["lr_grids"][arm]\n            if smoke:\n                grid = grid[2:5]\n            scores = []\n            for lr in grid:\n                curves = [rollout(arm, inputs, lr, gamma, data["eval_every"], False)\n                          for inputs in tune_inputs]\n                score = median(curve[-1][2] for curve in curves) if all(curves) else None\n                scores.append({"lr": lr, "median_final_loss": score})\n            valid = [row for row in scores if row["median_final_loss"] is not None]\n            if not valid:\n                raise RuntimeError(f"No finite tuning trial for {arm} at rotation {fraction}")\n            pick = min(valid, key=lambda row: row["median_final_loss"])["lr"]\n            lr_by_arm[arm] = pick\n            tuning.append(dict(rotation=fraction, optimizer=arm, selected_lr=pick,\n                               edge_pick=pick in (grid[0], grid[-1]), trials=scores))\n            print(f"rotation={fraction:g} {arm}: lr={pick:.4g}", flush=True)\n\n        regime_trials = []\n        for seed in eval_seeds:\n            inputs = problem(data, fraction, seed, steps, device)\n            order = np.random.Generator(np.random.PCG64(seed)).permutation(ARMS)\n            curves = {arm: rollout(arm, inputs, lr_by_arm[arm], gamma, data["eval_every"], True)\n                      for arm in order.tolist()}\n            # Shortest completed run defines the common observed time horizon.\n            budget = min((curve[-1][1] for curve in curves.values() if curve), default=None)\n            for arm in ARMS:\n                curve = curves[arm]\n                at_budget = next((p for p in reversed(curve) if p[1] <= budget), None) if curve else None\n                regime_trials.append(dict(rotation=fraction, off_diagonal=off_diagonal,\n                    seed=seed, optimizer=arm, lr=lr_by_arm[arm], completed=bool(curve),\n                    final_loss=curve[-1][2] if curve else None,\n                    elapsed_s=curve[-1][1] if curve else None, common_time_s=budget,\n                    loss_at_common_time=at_budget[2] if at_budget else None,\n                    steps_at_common_time=at_budget[0] if at_budget else None, curve=curve))\n        trials.extend(regime_trials)\n        for arm in ARMS:\n            rows = [r for r in regime_trials if r["optimizer"] == arm]\n            muon = {r["seed"]: r for r in regime_trials if r["optimizer"] == "Muon"}\n            result = dict(rotation=fraction, off_diagonal=round(off_diagonal, 4), optimizer=arm,\n                          completed=sum(r["completed"] for r in rows), total=len(rows))\n            for metric in ("final_loss", "loss_at_common_time"):\n                values = [r[metric] for r in rows if r[metric] is not None]\n                ratios = [r[metric] / muon[r["seed"]][metric] for r in rows\n                          if r[metric] is not None and muon[r["seed"]][metric]]\n                result[f"median_{metric}"] = median(values) if values else None\n                result[f"{metric}_paired_ratio_to_muon"] = median(ratios) if ratios else None\n            summary.append(result)\n        # Save each finished regime for inspection if the notebook is interrupted.\n        metadata = dict(device=str(device), torch_version=str(torch.__version__),\n                        gpu=torch.cuda.get_device_name(device) if device.type == "cuda" else None,\n                        steps=steps, smoke=smoke)\n        (out / "results.json").write_text(json.dumps(dict(\n            metadata=metadata, design=data, tuning=tuning, trials=trials, summary=summary),\n            indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n        with (out / "summary.csv").open("w", newline="", encoding="utf-8") as handle:\n            writer = csv.DictWriter(handle, fieldnames=list(summary[0]))\n            writer.writeheader()\n            writer.writerows(summary)\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--data", type=Path, required=True)\n    parser.add_argument("--sha256")\n    parser.add_argument("--out", type=Path, required=True)\n    parser.add_argument("--device", choices=("cpu", "cuda"), default="cpu")\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    raw = args.data.read_bytes()\n    if args.sha256 and hashlib.sha256(raw).hexdigest() != args.sha256:\n        parser.error("pilot dataset digest mismatch")\n    torch.set_num_threads(1)\n    torch.set_float32_matmul_precision("highest")\n    for row in run(json.loads(raw), args.out, torch.device(args.device), args.smoke):\n        print(row)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
Path('noise_geometry_data.json').write_bytes(b'{\n  "description": "Fixed matrix quadratic, rotated Kronecker Gaussian noise, trace 4",\n  "row_eigenvalues": [\n    0.08814783889856656,\n    0.0700736921953629,\n    0.05570553287801948,\n    0.04428347209924034,\n    0.03520343132626217,\n    0.02798519443926159,\n    0.02224701054124297,\n    0.017685404298203446,\n    0.014059126039027705,\n    0.011176392784039766,\n    0.00888474541848369,\n    0.007062985587263345,\n    0.005614765877490213,\n    0.004463494292821218,\n    0.003548283532518903,\n    0.00282073084475413\n  ],\n  "column_eigenvalues": [\n    1.0,\n    0.9047357242349298,\n    0.8185467307069028,\n    0.7405684692262438,\n    0.6700187503509588,\n    0.6061898993497573,\n    0.5484416576121018,\n    0.4961947603002903,\n    0.44892512582186056,\n    0.40615859883769795,\n    0.36746619407366893,\n    0.33245979322709407,\n    0.30078825180430996,\n    0.2721338768375308,\n    0.24620924014946255,\n    0.22275429519995577,\n    0.20153376859417332,\n    0.18233480008684413,\n    0.16496480740980207,\n    0.14924955450518296,\n    0.13503140378698733,\n    0.12216773489967922,\n    0.11052951411260215,\n    0.1\n  ],\n  "curvature": [\n    [\n      1.0,\n      0.9047357242349298,\n      0.8185467307069028,\n      0.7405684692262438,\n      0.6700187503509588,\n      0.6061898993497573,\n      0.5484416576121018,\n      0.4961947603002903,\n      0.44892512582186056,\n      0.40615859883769795,\n      0.36746619407366893,\n      0.33245979322709407,\n      0.30078825180430996,\n      0.2721338768375308,\n      0.24620924014946255,\n      0.22275429519995577,\n      0.20153376859417332,\n      0.18233480008684413,\n      0.16496480740980207,\n      0.14924955450518296,\n      0.13503140378698733,\n      0.12216773489967922,\n      0.11052951411260215,\n      0.1\n    ],\n    [\n      0.7356422544596414,\n      0.6655618278663601,\n      0.6021575623577949,\n      0.5447934582833195,\n      0.492894104038411,\n      0.44593890418831855,\n      0.4034568574453493,\n      0.3650218321183669,\n      0.33024829164317165,\n      0.2987874273171332,\n      0.2703236594460579,\n      0.2445714718067657,\n      0.22127254767229687,\n      0.20019317867160355,\n      0.18112192049234588,\n      0.16386747191146392,\n      0.14825675587836534,\n      0.13413318340233404,\n      0.12135508282944735,\n      0.10979427875328991,\n      0.09933480630470952,\n      0.08987174792382784,\n      0.08131018094612338,\n      0.07356422544596414\n    ],\n    [\n      0.5411695265464637,\n      0.4896154035338888,\n      0.4429725467128103,\n      0.4007730878664057,\n      0.36259372990468164,\n      0.3280515008283566,\n      0.2967999121882989,\n      0.26852548350654415,\n      0.24294459779582792,\n      0.21980065663577206,\n      0.19886150626867835,\n      0.1799171088964417,\n      0.1627774358196769,\n      0.1472705612854202,\n      0.13324093792304922,\n      0.12054783646955126,\n      0.10906393413323334,\n      0.09867403743594154,\n      0.08927392672279115,\n      0.08076931074884049,\n      0.0730748808563083,\n      0.06611345525491329,\n      0.05981520482172758,\n      0.05411695265464637\n    ],\n    [\n      0.3981071705534972,\n      0.360181779273837,\n      0.3258693229275405,\n      0.29482561788479456,\n      0.2667392689200102,\n      0.24132854564824113,\n      0.21833855652562376,\n      0.19753869206661934,\n      0.17872031163131363,\n      0.16169465057924887,\n      0.1462909267967306,\n      0.13235462760443914,\n      0.1197459598615467,\n      0.10833844771954329,\n      0.09801766396002903,\n      0.08868008219069286,\n      0.08023203838600959,\n      0.07258879135601107,\n      0.0656736727188189,\n      0.05941731785042835,\n      0.05375697009750431,\n      0.04863585127384103,\n      0.04400259212602088,\n      0.03981071705534972\n    ],\n    [\n      0.2928644564625237,\n      0.26496493612029043,\n      0.23972324337765283,\n      0.21688618221322709,\n      0.1962246771412329,\n      0.17753147538613862,\n      0.16061906795797373,\n      0.14531780877489675,\n      0.13147421296618925,\n      0.11894941728618261,\n      0.1076177871957373,\n      0.09736565663909591,\n      0.08809018787498195,\n      0.07969833992506282,\n      0.07210593529242332,\n      0.06523681558842759,\n      0.0590220775981766,\n      0.05339938212163652,\n      0.048312328657516584,\n      0.043709889657434206,\n      0.039545898675447604,\n      0.03577858727865224,\n      0.03237016607365407,\n      0.02928644564625237\n    ],\n    [\n      0.2154434690031884,\n      0.1949194029602853,\n      0.1763505472047138,\n      0.1595506400444829,\n      0.14435116387279182,\n      0.13059965479060534,\n      0.11815817326181012,\n      0.10690192046030009,\n      0.09671798642975446,\n      0.087504217499068,\n      0.0791681915926301,\n      0.07162629115692787,\n      0.06480286440412508,\n      0.05862946645916406,\n      0.0530441727984393,\n      0.04799095809323875,\n      0.04341913422721452,\n      0.039282841850712555,\n      0.035540590371810635,\n      0.03215484176977706,\n      0.02909163405623882,\n      0.026320240607048778,\n      0.023812861947655875,\n      0.02154434690031884\n    ],\n    [\n      0.15848931924611134,\n      0.14339094903163155,\n      0.12973091412086704,\n      0.11737219254280214,\n      0.10619081562525369,\n      0.0960746244818118,\n      0.0869221449611509,\n      0.0786415697734804,\n      0.07114983758398156,\n      0.06437179983574118,\n      0.05823946694469522,\n      0.05269132630526508,\n      0.04767172526569301,\n      0.04313031288378537,\n      0.03902153486339066,\n      0.035304176605388315,\n      0.031940949789593866,\n      0.02889811834063973,\n      0.026145160025945396,\n      0.023654460291311838,\n      0.021401035263046403,\n      0.01936228113808956,\n      0.01751774744830977,\n      0.015848931924611134\n    ],\n    [\n      0.11659144011798317,\n      0.10548444101473695,\n      0.09543554213698476,\n      0.08634394433305806,\n      0.07811845100946974,\n      0.07067655335016347,\n      0.0639436026816888,\n      0.05785206168240831,\n      0.05234082692471752,\n      0.04735461595478941,\n      0.042843412761723355,\n      0.038761966073673805,\n      0.035069335448435046,\n      0.031728480605377585,\n      0.0287058898793802,\n      0.02597124406982919,\n      0.023497112312799037,\n      0.02125867692574972,\n      0.019233484464694566,\n      0.017401220496726703,\n      0.01574350582867774,\n      0.014243712147905593,\n      0.012886795225929228,\n      0.011659144011798317\n    ],\n    [\n      0.08576958985908942,\n      0.07759881199849615,\n      0.07020641737322957,\n      0.06351825386810861,\n      0.05746723341550136,\n      0.05199265904395138,\n      0.04703961603502912,\n      0.042558421081185085,\n      0.038504123919181094,\n      0.034836056440051784,\n      0.03151742475277914,\n      0.028514940109725533,\n      0.025798484991688176,\n      0.023340811003118973,\n      0.021117265547137453,\n      0.019105544538650737,\n      0.017285468675078883,\n      0.015638781020487682,\n      0.014148963872722398,\n      0.012801073076561354,\n      0.011581588120906997,\n      0.010478276516359452,\n      0.009480071092762322,\n      0.008576958985908942\n    ],\n    [\n      0.06309573444801933,\n      0.05708496500196358,\n      0.051646807153977126,\n      0.046726711474875254,\n      0.04227532514733785,\n      0.03824799691444385,\n      0.03460432918892472,\n      0.03130777283040572,\n      0.02832526052589978,\n      0.025626875096043004,\n      0.02318554939989655,\n      0.020976794828100144,\n      0.018978455660928713,\n      0.017170486827250843,\n      0.015534752835119108,\n      0.014054845857092116,\n      0.012715921145526539,\n      0.01150454812691221,\n      0.010408575681597523,\n      0.009417010257544212,\n      0.008519905595487024,\n      0.007708262959346184,\n      0.00697394087111735,\n      0.006309573444801934\n    ],\n    [\n      0.046415888336127795,\n      0.04199411234979421,\n      0.03799357365039407,\n      0.034374143372862424,\n      0.03109951549940199,\n      0.02813684267870688,\n      0.02545640673860415,\n      0.02303132058706997,\n      0.0208372585114296,\n      0.01885221217040871,\n      0.01705626983142528,\n      0.015431416638680934,\n      0.01396135390856794,\n      0.012631335639768385,\n      0.011428020598100325,\n      0.010339338492393994,\n      0.009354368899026168,\n      0.008463231720621142,\n      0.0076569880801242,\n      0.0069275506561293915,\n      0.0062676025600473875,\n      0.005670523941381173,\n      0.005130325584897002,\n      0.00464158883361278\n    ],\n    [\n      0.034145488738336026,\n      0.030892643483034082,\n      0.02794967817515432,\n      0.025287072325931456,\n      0.02287811769458264,\n      0.020698650381540187,\n      0.018726808443628364,\n      0.016942812599854907,\n      0.015328767828106424,\n      0.013868483862590954,\n      0.012547312791461663,\n      0.011352002125585264,\n      0.010270561864607846,\n      0.009292144226875632,\n      0.008406934836797744,\n      0.007606054278166068,\n      0.006881469025926764,\n      0.006225910862972087,\n      0.0056328039736331685,\n      0.005096198982558394,\n      0.004610713277330281,\n      0.004171477016205018,\n      0.0037740842793856093,\n      0.0034145488738336026\n    ],\n    [\n      0.025118864315095794,\n      0.022725933898077127,\n      0.020560964264191947,\n      0.018602238894532212,\n      0.016830110078635777,\n      0.01522680183094813,\n      0.01377623158230461,\n      0.012463848857844474,\n      0.011276489323156622,\n      0.01020224273461356,\n      0.009230333469321148,\n      0.00835101243629618,\n      0.00755545928464733,\n      0.0068356939278229265,\n      0.006184496496437186,\n      0.005595334916732483,\n      0.005062299388226954,\n      0.004580043103301555,\n      0.004143728614092727,\n      0.003748979308704185,\n      0.003391835510002247,\n      0.0030687147566276353,\n      0.002776375867807919,\n      0.0025118864315095794\n    ],\n    [\n      0.018478497974222907,\n      0.016718157247482244,\n      0.015125514105174286,\n      0.013684592958370505,\n      0.012380940121051556,\n      0.011201478827128878,\n      0.010134378059164678,\n      0.009168933873028936,\n      0.008295462028077015,\n      0.007505200845835616,\n      0.006790223322785693,\n      0.006143357615657425,\n      0.005558115101635991,\n      0.00502862529185974,\n      0.004549576945336805,\n      0.004116164792601834,\n      0.00372404133470494,\n      0.003369273234035088,\n      0.0030483018595400995,\n      0.002757907590577695,\n      0.0024951775213343207,\n      0.0022574762418591235,\n      0.002042419402621561,\n      0.001847849797422291\n    ],\n    [\n      0.013593563908785255,\n      0.01229858288794863,\n      0.011126967296191517,\n      0.010066964815258211,\n      0.009107942702980192,\n      0.008240281137671028,\n      0.007455276722990228,\n      0.006745055185346377,\n      0.0061024923881189234,\n      0.00552114287040292,\n      0.004995175193458504,\n      0.0045193134463340345,\n      0.0040887843239136796,\n      0.0036992692465364707,\n      0.0033468610409051755,\n      0.0030280247477570154,\n      0.002739562163163234,\n      0.002478579757776099,\n      0.002242459652225596,\n      0.0020288333575239326,\n      0.0018355580170713996,\n      0.0016606949119503242,\n      0.001502490013896639,\n      0.0013593563908785257\n    ],\n    [\n      0.01,\n      0.009047357242349298,\n      0.008185467307069028,\n      0.0074056846922624375,\n      0.006700187503509588,\n      0.006061898993497573,\n      0.005484416576121019,\n      0.004961947603002903,\n      0.004489251258218606,\n      0.00406158598837698,\n      0.003674661940736689,\n      0.003324597932270941,\n      0.0030078825180430997,\n      0.002721338768375308,\n      0.0024620924014946257,\n      0.0022275429519995576,\n      0.0020153376859417334,\n      0.0018233480008684414,\n      0.0016496480740980208,\n      0.0014924955450518296,\n      0.0013503140378698733,\n      0.0012216773489967922,\n      0.0011052951411260215,\n      0.001\n    ]\n  ],\n  "rotations": [\n    0.0,\n    0.5,\n    1.0\n  ],\n  "gamma": 30,\n  "steps": 1200,\n  "eval_every": 20,\n  "tune_seeds": [\n    100,\n    101\n  ],\n  "evaluation_seeds": [\n    200,\n    201,\n    202,\n    203,\n    204,\n    205,\n    206,\n    207\n  ],\n  "lr_grids": {\n    "AdamW": [\n      0.0001,\n      0.0003162278,\n      0.001,\n      0.003162278,\n      0.01,\n      0.03162278,\n      0.1\n    ],\n    "Muon": [\n      0.0001,\n      0.0003162278,\n      0.001,\n      0.003162278,\n      0.01,\n      0.03162278,\n      0.1\n    ],\n    "Evie-KF": [\n      0.0001,\n      0.0003162278,\n      0.001,\n      0.003162278,\n      0.01,\n      0.03162278,\n      0.1\n    ],\n    "Evie diagonal": [\n      0.0001,\n      0.0003162278,\n      0.001,\n      0.003162278,\n      0.01,\n      0.03162278,\n      0.1\n    ],\n    "Evie random basis": [\n      0.0001,\n      0.0003162278,\n      0.001,\n      0.003162278,\n      0.01,\n      0.03162278,\n      0.1\n    ]\n  }\n}\n')


In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, 'pilot.py', '--data', 'noise_geometry_data.json',
    '--sha256', 'c21e5b0f0f7e743da219c3c5b43f8f53e4fc4959410cf6274a51c82a049f3720', '--out', 'noise_geometry_results', '--device', 'cpu',
], check=True)


In [ ]:
from pathlib import Path
import pandas as pd
summary = pd.read_csv(Path('noise_geometry_results') / 'summary.csv')
display(summary.round(5))
